# Financial Risk, Transaction & Customer Intelligence Platform

## Python Data Exploration & Intelligence

This notebook explores the cleaned financial analytics model and develops customer segmentation, transaction anomaly signals, and credit-risk intelligence.

**Data source:** `financial_analytics` MySQL database  
**Tools:** Python, pandas, NumPy, matplotlib, seaborn, scikit-learn, SQLAlchemy


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## 1. Connect to MySQL

Replace the password below with your local MySQL password. The notebook reads from the analytics layer rather than the raw files, keeping business analysis separate from data cleaning.


In [ ]:
connection_url = URL.create(
    "mysql+pymysql",
    username="root",
    password="YOUR_PASSWORD",
    host="localhost",
    port=3306,
    database="financial_analytics"
)

engine = create_engine(connection_url)


In [ ]:
transactions = pd.read_sql("SELECT * FROM fact_transactions", engine)
accounts = pd.read_sql("SELECT * FROM dim_account", engine)
customers = pd.read_sql("SELECT * FROM dim_customer", engine)
loans = pd.read_sql("SELECT * FROM fact_loans", engine)
loan_status = pd.read_sql("SELECT * FROM dim_loan_status", engine)
transaction_types = pd.read_sql("SELECT * FROM dim_transaction_type", engine)

print("Transactions:", transactions.shape)
print("Accounts:", accounts.shape)
print("Customers:", customers.shape)
print("Loans:", loans.shape)


## 2. Transaction activity overview

The transaction fact table contains 49,500 deduplicated transaction records. The analysis uses transaction **value**, not revenue or profit, because the dataset does not contain revenue/profit fields.


In [ ]:
transaction_summary = (
    transactions
    .merge(transaction_types, on="TransactionTypeID", how="left")
    .groupby("TypeName")
    .agg(
        Transactions=("TransactionID", "count"),
        TotalValue=("Amount", "sum"),
        AverageValue=("Amount", "mean")
    )
    .reset_index()
    .sort_values("TotalValue", ascending=False)
)

transaction_summary


In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=transaction_summary, x="TypeName", y="TotalValue")
plt.title("Transaction Value by Type")
plt.xlabel("Transaction Type")
plt.ylabel("Total Transaction Value")
plt.tight_layout()
plt.show()


## 3. Transaction amount distribution

Individual transaction amounts are broadly distributed across the available range. This supports using account-level behavioral patterns rather than treating a single large transaction amount as an anomaly by itself.


In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(transactions["Amount"], bins=40)
plt.title("Transaction Amount Distribution")
plt.xlabel("Transaction Amount")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


## 4. Account-level behavioral analysis

For each account acting as the transaction origin, calculate:
- transaction frequency
- total transaction value
- average transaction size

These measures establish a behavioral baseline for identifying accounts that are unusually active or have unusually high transaction values.


In [ ]:
account_activity = (
    transactions
    .groupby("AccountOriginID")
    .agg(
        TransactionCount=("TransactionID", "count"),
        TotalTransactionValue=("Amount", "sum"),
        AverageTransactionValue=("Amount", "mean")
    )
    .reset_index()
    .rename(columns={"AccountOriginID": "AccountID"})
)

account_activity.head()


In [ ]:
baseline = account_activity[
    ["TransactionCount", "TotalTransactionValue", "AverageTransactionValue"]
].agg(["mean", "std"])

frequency_threshold = baseline.loc["mean", "TransactionCount"] + 2 * baseline.loc["std", "TransactionCount"]
value_threshold = baseline.loc["mean", "TotalTransactionValue"] + 2 * baseline.loc["std", "TotalTransactionValue"]
average_size_threshold = baseline.loc["mean", "AverageTransactionValue"] + 2 * baseline.loc["std", "AverageTransactionValue"]

print(f"Frequency threshold: {frequency_threshold:,.2f}")
print(f"Total value threshold: {value_threshold:,.2f}")
print(f"Average size threshold: {average_size_threshold:,.2f}")


In [ ]:
account_activity["FrequencyFlag"] = (
    account_activity["TransactionCount"] > frequency_threshold
).astype(int)

account_activity["ValueFlag"] = (
    account_activity["TotalTransactionValue"] > value_threshold
).astype(int)

account_activity["AverageSizeFlag"] = (
    account_activity["AverageTransactionValue"] > average_size_threshold
).astype(int)

account_activity["AnomalyScore"] = (
    account_activity["FrequencyFlag"]
    + account_activity["ValueFlag"]
    + account_activity["AverageSizeFlag"]
)

account_activity["AnomalyCategory"] = pd.cut(
    account_activity["AnomalyScore"],
    bins=[-1, 0, 1, 2, 3],
    labels=[
        "Normal",
        "Low Anomaly Signal",
        "Moderate Anomaly Signal",
        "High Anomaly Signal"
    ]
)

account_activity.sort_values(
    ["AnomalyScore", "TotalTransactionValue"],
    ascending=[False, False]
).head(20)


### Interpretation

Anomaly scores indicate **accounts requiring investigation**, not confirmed fraud. The dataset does not contain a fraud label, so the analysis deliberately avoids claiming fraud detection.


## 5. Customer segmentation

Customer behavior is summarized using account and transaction metrics:
- account count
- total balance
- average balance
- transaction count
- total transaction value
- average transaction value

K-Means clustering is tested across multiple values of K. Customers with no active financial relationship are retained as a separate behavioral group after the account-level features are prepared.


In [ ]:
customer_accounts = (
    accounts
    .groupby("CustomerID")
    .agg(
        AccountCount=("AccountID", "count"),
        TotalBalance=("Balance", "sum"),
        AverageBalance=("Balance", "mean")
    )
    .reset_index()
)

customer_transactions = (
    transactions
    .merge(
        accounts[["AccountID", "CustomerID"]],
        left_on="AccountOriginID",
        right_on="AccountID",
        how="left"
    )
    .groupby("CustomerID")
    .agg(
        TransactionCount=("TransactionID", "count"),
        TotalTransactionValue=("Amount", "sum"),
        AverageTransactionValue=("Amount", "mean")
    )
    .reset_index()
)

customer_features = customers.merge(customer_accounts, on="CustomerID", how="left")
customer_features = customer_features.merge(customer_transactions, on="CustomerID", how="left")

account_cols = ["AccountCount", "TotalBalance", "AverageBalance"]
transaction_cols = ["TransactionCount", "TotalTransactionValue", "AverageTransactionValue"]

customer_features[account_cols] = customer_features[account_cols].fillna(0)
customer_features[transaction_cols] = customer_features[transaction_cols].fillna(0)

feature_cols = account_cols + transaction_cols
X = customer_features[feature_cols]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
silhouette_results = []

for k in range(2, 9):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_results.append({"K": k, "SilhouetteScore": score})

silhouette_df = pd.DataFrame(silhouette_results)
silhouette_df


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(silhouette_df["K"], silhouette_df["SilhouetteScore"], marker="o")
plt.title("K-Means Silhouette Scores")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.xticks(range(2, 9))
plt.tight_layout()
plt.show()


The analysis selects **K=3** as the business-friendly segmentation:
- **Core Active**
- **High-Value Active**
- **No Financial Relationship**

The segmentation is driven primarily by financial behavior rather than customer type.


In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
customer_features["Cluster_3"] = kmeans.fit_predict(X_scaled)

cluster_profile = (
    customer_features
    .groupby("Cluster_3")
    .agg(
        Customers=("CustomerID", "count"),
        AverageBalance=("TotalBalance", "mean"),
        AverageTransactions=("TransactionCount", "mean"),
        AverageTransactionValue=("TotalTransactionValue", "mean")
    )
    .reset_index()

cluster_profile


In [ ]:
segment_mapping = {
    0: "Core Active",
    1: "No Financial Relationship",
    2: "High-Value Active"
}

customer_features["CustomerSegment"] = customer_features["Cluster_3"].map(segment_mapping)

segment_contribution = (
    customer_features
    .groupby("CustomerSegment")
    .agg(
        Customers=("CustomerID", "count"),
        TotalBalance=("TotalBalance", "sum"),
        TotalTransactionValue=("TotalTransactionValue", "sum")
    )
    .reset_index()
)

segment_contribution["CustomerShare"] = (
    segment_contribution["Customers"] / segment_contribution["Customers"].sum()
)

segment_contribution["BalanceShare"] = (
    segment_contribution["TotalBalance"] / segment_contribution["TotalBalance"].sum()
)

segment_contribution


## 6. Loan portfolio and credit exposure

Loan analysis is kept separate from transaction anomaly analysis. Overdue exposure is calculated from the deduplicated loan fact table.


In [ ]:
loans_enriched = loans.merge(
    loan_status,
    on="LoanStatusID",
    how="left"
)

loan_portfolio_summary = (
    loans_enriched
    .groupby("LoanStatus")
    .agg(
        Loans=("LoanID", "count"),
        PrincipalExposure=("PrincipalAmount", "sum"),
        AverageInterestRate=("InterestRate", "mean")
    )
    .reset_index()
)

loan_portfolio_summary


In [ ]:
total_loan_exposure = loans_enriched["PrincipalAmount"].sum()

overdue_exposure = loans_enriched.loc[
    loans_enriched["LoanStatus"] == "Overdue",
    "PrincipalAmount"
].sum()

overdue_exposure_pct = overdue_exposure / total_loan_exposure

print(f"Total loan exposure: {total_loan_exposure:,.2f}")
print(f"Overdue exposure: {overdue_exposure:,.2f}")
print(f"Overdue exposure %: {overdue_exposure_pct:.2%}")


## 7. Account-level risk intelligence

Loan records can contain multiple loans per account, so risk prioritization is aggregated to **account grain**. The maximum anomaly score is retained so the score remains on the defined 0–3 scale.


In [ ]:
loan_customer_analysis = (
    loans_enriched
    .merge(
        accounts[["AccountID", "CustomerID", "AccountTypeID", "AccountStatusID", "Balance"]],
        on="AccountID",
        how="left"
    )
    .merge(
        customers[["CustomerID", "CustomerTypeID"]],
        on="CustomerID",
        how="left"
    )
)

risk_intelligence = loan_customer_analysis.merge(
    account_activity,
    on="AccountID",
    how="left"
)

risk_intelligence[[
    "TransactionCount",
    "TotalTransactionValue",
    "AverageTransactionValue",
    "AnomalyScore"
]] = risk_intelligence[[
    "TransactionCount",
    "TotalTransactionValue",
    "AverageTransactionValue",
    "AnomalyScore"
]].fillna(0)

account_risk = (
    risk_intelligence
    .groupby("AccountID")
    .agg(
        CustomerID=("CustomerID", "first"),
        TotalLoanExposure=("PrincipalAmount", "sum"),
        OverdueLoanExposure=(
            "PrincipalAmount",
            lambda x: x[
                risk_intelligence.loc[x.index, "LoanStatus"] == "Overdue"
            ].sum()
        ),
        AverageInterestRate=("InterestRate", "mean"),
        TransactionCount=("TransactionCount", "first"),
        TotalTransactionValue=("TotalTransactionValue", "first"),
        AverageTransactionValue=("AverageTransactionValue", "first"),
        AnomalyScore=("AnomalyScore", "max"),
        AnomalyCategory=("AnomalyCategory", "first")
    )
    .reset_index()
)

account_risk.sort_values(
    ["AnomalyScore", "TotalLoanExposure"],
    ascending=[False, False]
).head(20)


## 8. Key business takeaways

1. Transaction value is distributed across deposits, transfers and withdrawals, with payments representing a smaller share.
2. Individual transaction amount alone is not a strong risk signal; account-level behavior is more informative.
3. Customer segmentation separates a broad core-active population, a smaller high-value group, and customers with no observed financial relationship.
4. High-value customers contribute a disproportionate share of overall balance and transaction value.
5. Overdue loans represent roughly 9.8% of deduplicated principal exposure.
6. Account-level anomaly scoring provides a prioritization layer for investigation, but it is **not fraud detection**.
7. Transaction data after January 2024 contains a major coverage gap, so the later sparse records should not be interpreted as a sustained business decline.


## 9. Export analytical outputs

The following outputs support the Power BI risk and executive dashboards.


In [ ]:
output_dir = "../data/processed"
os.makedirs(output_dir, exist_ok=True)

customer_features.to_csv(
    f"{output_dir}/customer_intelligence.csv", index=False
)

account_activity.to_csv(
    f"{output_dir}/account_anomaly_analysis.csv", index=False
)

risk_intelligence.to_csv(
    f"{output_dir}/risk_intelligence.csv", index=False
)

account_risk.to_csv(
    f"{output_dir}/account_risk_intelligence.csv", index=False
)

segment_contribution.to_csv(
    f"{output_dir}/customer_segment_contribution.csv", index=False
)

loan_portfolio_summary.to_csv(
    f"{output_dir}/loan_portfolio_summary.csv", index=False
)

print("Analytical outputs exported successfully.")
